In [1]:
import os, time, json, csv, subprocess
import torch, torch.nn as nn

RESULTS_DIR = '/kaggle/working/results_v2'
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS      = ['resnet18', 'resnet34', 'resnet50', 'vittiny', 'vitbase']
BATCH_SIZES = [16, 32, 64]      # GLOBAL batch: DP gives each GPU B/2
RESOLUTION  = 224
NUM_CLASSES = 200
WARMUP, MEASURED = 20, 100
REPEAT = 1 # change to 2, then 3, for the repeats
WARM_SEC = 60

def run_path(model, config, bs, res=RESOLUTION, repeat=REPEAT):
    return f"{RESULTS_DIR}/{model}-{config}-bs{bs}-res{res}-r{repeat}.csv"

def already_done(path):
    if os.path.exists(path):
        print(f"skip (exists): {os.path.basename(path)}")
        return True
    return False

In [2]:
   print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
         "| GPUs:", torch.cuda.device_count())

2.11.0+cu128 | CUDA available: True | GPUs: 2


In [3]:
def nccl_version():
    try:
        return str(torch.cuda.nccl.version())
    except Exception as e:
        return f"unavailable ({type(e).__name__})"

info = {
    'python': __import__('sys').version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'nccl': nccl_version(),
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'p2p_0_to_1': torch.cuda.device_count() > 1 and torch.cuda.can_device_access_peer(0, 1),
    'topology': subprocess.run(['nvidia-smi', 'topo', '-m'],
                               capture_output=True, text=True).stdout,
}
json.dump(info, open(f'{RESULTS_DIR}/environment-r{REPEAT}.json', 'w'), indent=2)
print(json.dumps(info, indent=2))
assert len(info['gpus']) == 2, "Need 2 GPUs: set Accelerator to GPU T4 x2"

{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "nccl": "(2, 28, 9)",
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "p2p_0_to_1": true,
  "topology": "\t\u001bGPU0\tGPU1\tCPU Affinity\tNUMA Affinity\tGPU NUMA ID\u001b\nGPU0\t X \tPHB\t0-3\t0\t\tN/A\nGPU1\tPHB\t X \t0-3\t0\t\tN/A\n\nLegend:\n\n  X    = Self\n  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)\n  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node\n  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)\n  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)\n  PIX  = Connection traversing at most a single PCIe bridge\n  NV#  = Connection traversing a bonded set of # NVLinks\n"
}


In [4]:
from torchvision import models
from torchvision.models.vision_transformer import VisionTransformer

def build_model(name, res=RESOLUTION):
    if name == 'resnet18': return models.resnet18(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet34': return models.resnet34(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet50': return models.resnet50(weights=None, num_classes=NUM_CLASSES)
    if name == 'vitbase':  return models.vit_b_16(weights=None, image_size=res,
                                                  num_classes=NUM_CLASSES)
    if name == 'vittiny':  return VisionTransformer(image_size=res, patch_size=16,
                               num_layers=12, num_heads=3, hidden_dim=192,
                               mlp_dim=768, num_classes=NUM_CLASSES)
    raise ValueError(name)

def make_batch(bs, device, res=RESOLUTION, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(bs, 3, res, res, generator=g).to(device)
    y = torch.randint(0, NUM_CLASSES, (bs,), generator=g).to(device)
    return x, y

for m in MODELS:
    n = sum(p.numel() for p in build_model(m).parameters()) / 1e6
    print(f"{m:10s} {n:6.2f}M params")

resnet18    11.28M params
resnet34    21.39M params
resnet50    23.92M params
vittiny      5.56M params
vitbase     85.95M params


In [5]:
def start_clock_log(tag):
    f = open(f"{RESULTS_DIR}/{tag}-clocks.csv", 'w')
    p = subprocess.Popen(
        ['nvidia-smi', '--query-gpu=timestamp,index,clocks.sm,power.draw,temperature.gpu',
         '--format=csv,noheader,nounits', '-lms', '200'], stdout=f)
    return p, f

def stop_clock_log(p, f):
    p.terminate(); p.wait(); f.close()

In [6]:
def bench_1gpu(model_name, bs, device='cuda:0'):
    path = run_path(model_name, '1gpu', bs)
    if already_done(path): return
    torch.cuda.empty_cache()

    model = build_model(model_name).to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, device)

    def fwd_bwd():
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()

    # warm up for at least WARMUP steps AND at least WARM_SEC seconds
    t_w, n = time.perf_counter(), 0
    while n < WARMUP or time.perf_counter() - t_w < WARM_SEC:
        fwd_bwd(); opt.step(); n += 1
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)

    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows = []
    for i in range(MEASURED):
        torch.cuda.synchronize(device); t0 = time.perf_counter()
        fwd_bwd()
        torch.cuda.synchronize(device); t1 = time.perf_counter()
        opt.step()
        torch.cuda.synchronize(device); t2 = time.perf_counter()
        rows.append([model_name, '1gpu', bs, RESOLUTION, REPEAT, i,
                     f"{(t1-t0)*1e3:.3f}", f"{(t2-t1)*1e3:.3f}", f"{(t2-t0)*1e3:.3f}"])
    stop_clock_log(p, f)
    peak = torch.cuda.max_memory_allocated(device) / 2**20

    tmp = path + '.tmp'
    with open(tmp, 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['model', 'config', 'batch', 'res', 'repeat', 'step',
                    'fwdbwd_ms', 'opt_ms', 'total_ms', 'peak_mem_mb'])
        for r in rows: w.writerow(r + [f"{peak:.1f}"])
    os.replace(tmp, path)

    med = sorted(float(r[8]) for r in rows)[MEASURED // 2]
    print(f"{tag}: median {med:.1f} ms | peak {peak:.0f} MB | warm-up {n} steps")
    del model, opt, x, y
    torch.cuda.empty_cache()

In [7]:
for m in MODELS:
    for bs in [8] + BATCH_SIZES:
        bench_1gpu(m, bs)

resnet18-1gpu-bs8-res224-r1: median 31.9 ms | peak 344 MB | warm-up 1947 steps
resnet18-1gpu-bs16-res224-r1: median 51.1 ms | peak 515 MB | warm-up 1147 steps
resnet18-1gpu-bs32-res224-r1: median 98.3 ms | peak 856 MB | warm-up 618 steps
resnet18-1gpu-bs64-res224-r1: median 190.6 ms | peak 1634 MB | warm-up 316 steps
resnet34-1gpu-bs8-res224-r1: median 58.1 ms | peak 517 MB | warm-up 1041 steps
resnet34-1gpu-bs16-res224-r1: median 88.5 ms | peak 761 MB | warm-up 683 steps
resnet34-1gpu-bs32-res224-r1: median 164.2 ms | peak 1246 MB | warm-up 364 steps
resnet34-1gpu-bs64-res224-r1: median 319.0 ms | peak 2237 MB | warm-up 189 steps
resnet50-1gpu-bs8-res224-r1: median 92.9 ms | peak 925 MB | warm-up 652 steps
resnet50-1gpu-bs16-res224-r1: median 173.4 ms | peak 1608 MB | warm-up 350 steps
resnet50-1gpu-bs32-res224-r1: median 340.1 ms | peak 2979 MB | warm-up 178 steps
resnet50-1gpu-bs64-res224-r1: median 673.5 ms | peak 5525 MB | warm-up 91 steps
vittiny-1gpu-bs8-res224-r1: median 36.0 m

In [8]:
import pandas as pd

rows = []
for m in MODELS:
    for bs in BATCH_SIZES:
        p = run_path(m, '1gpu', bs)
        if not os.path.exists(p):
            print("missing:", os.path.basename(p)); continue
        d = pd.read_csv(p)
        q1, med, q3 = d.total_ms.quantile([.25, .5, .75])
        rows.append(dict(model=m, batch=bs, n=len(d), median_ms=med,
                         iqr_pct=100 * (q3 - q1) / med,
                         opt_ms=d.opt_ms.median(), peak_mb=d.peak_mem_mb.iloc[0]))
summary_1gpu = pd.DataFrame(rows)
print(summary_1gpu.round(2), "\n")

for m, g in summary_1gpu.groupby('model', sort=False):
    g = g.set_index('batch')
    checks = {
        'all steps recorded':       (g.n == MEASURED).all(),
        'stable (IQR < 10%)':       (g.iqr_pct < 10).all(),
        'optimizer time flat':      g.opt_ms.max() / g.opt_ms.min() < 1.3,
        'memory grows with batch':  g.peak_mb.is_monotonic_increasing,
        'time scales (64/32 ≈ 2x)': 1.5 <= g.loc[64, 'median_ms'] / g.loc[32, 'median_ms'] <= 2.2,
    }
    for k, v in checks.items():
        print(f"{m:9s} {'PASS' if v else 'WARN'}  {k}")

       model  batch    n  median_ms  iqr_pct  opt_ms  peak_mb
0   resnet18     16  100      51.14     1.43    1.75    514.6
1   resnet18     32  100      98.29     1.50    1.77    855.9
2   resnet18     64  100     190.63     0.89    1.84   1633.6
3   resnet34     16  100      88.48     1.35    3.28    761.1
4   resnet34     32  100     164.19     1.02    3.32   1245.7
5   resnet34     64  100     319.00     0.51    3.45   2237.0
6   resnet50     16  100     173.38     1.12    3.82   1608.2
7   resnet50     32  100     340.10     1.02    3.90   2978.6
8   resnet50     64  100     673.44     1.03    4.09   5524.6
9    vittiny     16  100      66.81     1.10    1.33    557.9
10   vittiny     32  100     132.82     1.84    1.40   1058.2
11   vittiny     64  100     263.04     1.03    1.48   2053.6
12   vitbase     16  100     600.28     0.64   12.25   2651.8
13   vitbase     32  100    1180.76     0.79   12.32   4604.5
14   vitbase     64  100    2373.85     0.43   12.42   8476.2 

resnet

In [9]:
%%writefile /kaggle/working/dp_bench.py
import os, time, csv
import torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.profiler import profile, ProfilerActivity
from torchvision import models
from torchvision.models.vision_transformer import VisionTransformer

NUM_CLASSES = 200

def build_model(name, res):
    if name == 'resnet18': return models.resnet18(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet34': return models.resnet34(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet50': return models.resnet50(weights=None, num_classes=NUM_CLASSES)
    if name == 'vitbase':  return models.vit_b_16(weights=None, image_size=res,
                                                  num_classes=NUM_CLASSES)
    if name == 'vittiny':  return VisionTransformer(image_size=res, patch_size=16,
                               num_layers=12, num_heads=3, hidden_dim=192,
                               mlp_dim=768, num_classes=NUM_CLASSES)
    raise ValueError(name)

def make_batch(bs, device, res, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(bs, 3, res, res, generator=g).to(device)
    y = torch.randint(0, NUM_CLASSES, (bs,), generator=g).to(device)
    return x, y

Writing /kaggle/working/dp_bench.py


In [10]:
%%writefile -a /kaggle/working/dp_bench.py

def dp_worker(rank, world, model_name, global_bs, res, repeat, warmup, measured,
              path, port, warm_sec):
    os.environ['MASTER_ADDR'], os.environ['MASTER_PORT'] = 'localhost', str(port)
    dist.init_process_group('nccl', rank=rank, world_size=world)
    torch.cuda.set_device(rank)
    dev, bs = f'cuda:{rank}', global_bs // world          # per-GPU batch = B / 2

    model = DDP(build_model(model_name, res).to(dev), device_ids=[rank])
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, dev, res, seed=rank)

    def nosync():
        opt.zero_grad(set_to_none=True)
        with model.no_sync():
            crit(model(x), y).backward()
    def synced():
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()
    def timed(fn):
        torch.cuda.synchronize(dev); t = time.perf_counter()
        fn(); torch.cuda.synchronize(dev)
        return (time.perf_counter() - t) * 1e3

    # warm up for at least `warmup` steps AND `warm_sec` seconds; GPU 0 decides when to stop
    t_w, n = time.perf_counter(), 0
    while True:
        nosync(); synced(); opt.step(); n += 1
        done = torch.tensor([float(n >= warmup and time.perf_counter() - t_w >= warm_sec)],
                            device=dev)
        dist.broadcast(done, src=0)
        if done.item(): break
    torch.cuda.synchronize(dev)
    torch.cuda.reset_peak_memory_stats(dev)

    rows = []
    for i in range(measured):
        if i % 2 == 0:
            t_ns = timed(nosync); t_s = timed(synced); t_o = timed(opt.step)
        else:
            t_s = timed(synced); t_o = timed(opt.step); t_ns = timed(nosync)
        rows.append([model_name, 'dp', global_bs, bs, res, repeat, i, i % 2,
                     round(t_ns, 3), round(t_s, 3), round(t_o, 3),
                     round(t_s + t_o, 3), round(t_s - t_ns, 3)])

    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        for _ in range(20):
            synced(); opt.step()
        torch.cuda.synchronize(dev)
    dt = lambda e: getattr(e, 'device_time_total', 0) or getattr(e, 'cuda_time_total', 0)
    nccl = [e for e in prof.key_averages()
            if e.key.lower().startswith('nccl') and 'kernel' in e.key.lower()]
    nccl_ms = sum(dt(e) for e in nccl) / 1e3 / 20

    if rank == 0:
        peak = torch.cuda.max_memory_allocated(dev) / 2**20
        tmp = path + '.tmp'
        with open(tmp, 'w', newline='') as fh:
            w = csv.writer(fh)
            w.writerow(['model', 'config', 'batch', 'per_gpu_batch', 'res', 'repeat', 'step',
                        'order', 'nosync_ms', 'sync_ms', 'opt_ms', 'step_ms',
                        'exposed_comm_ms', 'nccl_ms', 'peak_mem_mb'])
            for r in rows: w.writerow(r + [round(nccl_ms, 3), round(peak, 1)])
        os.replace(tmp, path)
        med = lambda k: sorted(r[k] for r in rows)[measured // 2]
        print(f"{os.path.basename(path)[:-4]}: step {med(11):.1f} ms | "
              f"exposed {med(12):.1f} ms | nccl {nccl_ms:.1f} ms | warm-up {n} steps")

    dist.destroy_process_group()

Appending to /kaggle/working/dp_bench.py


In [11]:
import sys, socket, importlib
sys.path.append('/kaggle/working')
import dp_bench; importlib.reload(dp_bench)
import torch.multiprocessing as mp

def free_port():
    s = socket.socket(); s.bind(('', 0)); port = s.getsockname()[1]; s.close()
    return port

def run_dp(m, bs):
    path = run_path(m, 'dp', bs)
    if already_done(path): return
    p, f = start_clock_log(os.path.basename(path)[:-4])
    try:
        mp.spawn(dp_bench.dp_worker,
                 args=(2, m, bs, RESOLUTION, REPEAT, WARMUP, MEASURED, path,
                       free_port(), WARM_SEC),
                 nprocs=2, join=True)
    finally:
        stop_clock_log(p, f)

for m in MODELS:
    for bs in BATCH_SIZES:
        run_dp(m, bs)

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


resnet18-dp-bs16-res224-r1: step 38.1 ms | exposed 6.5 ms | nccl 10.4 ms | warm-up 876 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


resnet18-dp-bs32-res224-r1: step 56.6 ms | exposed 5.2 ms | nccl 10.2 ms | warm-up 553 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


...

vitbase-dp-bs16-res224-r1: step 315.4 ms | exposed 19.0 ms | nccl 79.7 ms | warm-up 98 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


vitbase-dp-bs32-res224-r1: step 602.2 ms | exposed 14.1 ms | nccl 79.4 ms | warm-up 50 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


vitbase-dp-bs64-res224-r1: step 1176.2 ms | exposed 14.9 ms | nccl 79.7 ms | warm-up 26 steps


In [12]:
rows = []
for m in MODELS:
    for bs in BATCH_SIZES:
        p = run_path(m, 'dp', bs)
        if not os.path.exists(p): print("missing:", os.path.basename(p)); continue
        d = pd.read_csv(p)
        one  = pd.read_csv(run_path(m, '1gpu', bs))
        half = pd.read_csv(run_path(m, '1gpu', bs // 2))
        t1, th, tdp = one.total_ms.median(), half.total_ms.median(), d.step_ms.median()
        exp, nccl = d.exposed_comm_ms.median(), d.nccl_ms.iloc[0]
        rows.append(dict(model=m, batch=bs,
            per_gpu_ok   = (d.per_gpu_batch == bs // 2).all(),
            speedup      = t1 / tdp,
            ceiling      = t1 / th,
            overhead_eff = th / tdp,
            conc_pen_pct = 100 * (d.nosync_ms.median() / half.fwdbwd_ms.median() - 1),
            nccl_ms      = nccl,
            exposed_ms   = exp,
            overlap_pct  = 100 * (1 - exp / nccl),
            neg_exp_pct  = 100 * (d.exposed_comm_ms < 0).mean(),
            order_gap_ms = d[d.order == 1].exposed_comm_ms.median()
                         - d[d.order == 0].exposed_comm_ms.median(),
            iqr_pct      = 100 * (d.step_ms.quantile(.75) - d.step_ms.quantile(.25)) / tdp))
summary_dp = pd.DataFrame(rows)
print(summary_dp.round(2), "\n")

for m, g in summary_dp.groupby('model', sort=False):
    checks = {
        'per-GPU batch = B/2':      g.per_gpu_ok.all(),
        'speedup within ceiling':   (g.speedup <= g.ceiling * 1.03).all(),
        'NCCL ~flat across batch':  g.nccl_ms.max() / g.nccl_ms.min() < 1.35,
        'stable (IQR < 10%)':       (g.iqr_pct < 10).all(),
        'pass order effect < 2 ms': (g.order_gap_ms.abs() < 2).all(),
    }
    for k, v in checks.items():
        print(f"{m:9s} {'PASS' if v else 'WARN'}  {k}")

       model  batch  per_gpu_ok  speedup  ceiling  overhead_eff  conc_pen_pct  \
0   resnet18     16        True     1.34     1.60          0.84         -1.62   
1   resnet18     32        True     1.74     1.92          0.90          0.33   
2   resnet18     64        True     1.89     1.94          0.98         -1.71   
3   resnet34     16        True     1.31     1.52          0.86         -1.59   
4   resnet34     32        True     1.71     1.86          0.92         -2.67   
5   resnet34     64        True     1.90     1.94          0.98         -2.84   
6   resnet50     16        True     1.73     1.87          0.93         -1.76   
7   resnet50     32        True     1.99     1.96          1.02         -4.44   
8   resnet50     64        True     2.01     1.98          1.01         -3.05   
9    vittiny     16        True     1.59     1.85          0.86         -5.41   
10   vittiny     32        True     1.87     1.99          0.94         -3.04   
11   vittiny     64        T

In [13]:
def clock_stats(tag, gpu=0, min_power=35):
    c = pd.read_csv(f"{RESULTS_DIR}/{tag}-clocks.csv", header=None, skipinitialspace=True,
                    names=['ts', 'gpu', 'sm_mhz', 'power_w', 'temp_c'])
    c = c[(c.gpu == gpu) & (c.power_w > min_power)]      # keep only samples where the GPU is busy
    return c[['sm_mhz', 'power_w', 'temp_c']].median()

rows = []
for m in MODELS:
    for bs in [32, 64]:
        one = clock_stats(f"{m}-1gpu-bs{bs // 2}-res224-r{REPEAT}")   # same per-GPU work
        dp  = clock_stats(f"{m}-dp-bs{bs}-res224-r{REPEAT}")
        rows.append(dict(model=m, dp_batch=bs,
                         mhz_1gpu=one.sm_mhz, mhz_dp=dp.sm_mhz,
                         clock_drop_pct=100 * (1 - dp.sm_mhz / one.sm_mhz),
                         temp_1gpu=one.temp_c, temp_dp=dp.temp_c,
                         power_1gpu=one.power_w, power_dp=dp.power_w))
clocks = pd.DataFrame(rows).merge(summary_dp[['model', 'batch', 'conc_pen_pct']],
                                  left_on=['model', 'dp_batch'], right_on=['model', 'batch'])
print(clocks.drop(columns='batch').round(1))

      model  dp_batch  mhz_1gpu  mhz_dp  clock_drop_pct  temp_1gpu  temp_dp  \
0  resnet18        32     945.0   930.0             1.6       76.0     78.0   
1  resnet18        64     847.5   855.0            -0.9       77.0     77.0   
2  resnet34        32     870.0   915.0            -5.2       77.0     76.0   
3  resnet34        64     832.5   870.0            -4.5       77.0     75.0   
4  resnet50        32     780.0   840.0            -7.7       77.0     75.0   
5  resnet50        64     720.0   750.0            -4.2       77.0     76.0   
6   vittiny        32    1005.0  1020.0            -1.5       77.0     76.0   
7   vittiny        64     930.0   960.0            -3.2       77.0     75.0   
8   vitbase        32     795.0   810.0            -1.9       77.0     76.0   
9   vitbase        64     750.0   765.0            -2.0       77.0     76.0   

   power_1gpu  power_dp  conc_pen_pct  
0        66.2      66.3           0.3  
1        66.5      66.8          -1.7  
2        6

In [14]:
%%writefile -a /kaggle/working/dp_bench.py

def allreduce_worker(rank, world, sizes_mb, iters, path, port):
    os.environ['NCCL_DEBUG'], os.environ['NCCL_DEBUG_SUBSYS'] = 'INFO', 'INIT'   # prints transport
    os.environ['MASTER_ADDR'], os.environ['MASTER_PORT'] = 'localhost', str(port)
    dist.init_process_group('nccl', rank=rank, world_size=world)
    torch.cuda.set_device(rank)
    dev = f'cuda:{rank}'
    rows = []
    for mb in sizes_mb:
        t = torch.ones(int(mb * 2**20 / 4), device=dev)
        for _ in range(5): dist.all_reduce(t)
        torch.cuda.synchronize(dev)
        times = []
        for _ in range(iters):
            dist.barrier(device_ids=[rank]); torch.cuda.synchronize(dev)   # start together
            s = time.perf_counter()
            dist.all_reduce(t); torch.cuda.synchronize(dev)
            times.append((time.perf_counter() - s) * 1e3)
        med = sorted(times)[iters // 2]
        rows.append([round(mb, 2), round(med, 3), round(mb / 1024 / (med / 1e3), 3)])
    if rank == 0:
        with open(path, 'w', newline='') as fh:
            w = csv.writer(fh); w.writerow(['size_mb', 'median_ms', 'gb_per_s']); w.writerows(rows)
        for r in rows: print(f"all-reduce {r[0]:8.2f} MB: {r[1]:8.3f} ms  ->  {r[2]:.2f} GB/s")
    dist.destroy_process_group()

Appending to /kaggle/working/dp_bench.py


In [15]:
importlib.reload(dp_bench)
grad_mb = sorted(sum(p.numel() for p in build_model(m).parameters()) * 4 / 2**20 for m in MODELS)
sizes = sorted(set([1, 4, 16, 64, 256] + grad_mb))     # a sweep plus each model's exact gradient size
ar_path = f"{RESULTS_DIR}/allreduce-r{REPEAT}.csv"
if not already_done(ar_path):
    mp.spawn(dp_bench.allreduce_worker, args=(2, sizes, 50, ar_path, free_port()),
             nprocs=2, join=True)

49651bd10af9:650:650 [1] NCCL INFO ENV/Plugin: Could not find: libnccl-env.so
49651bd10af9:650:650 [1] NCCL INFO cudaDriverVersion 13000
49651bd10af9:650:650 [1] NCCL INFO NCCL version 2.31.2+cuda13.3
49651bd10af9:650:650 [1] NCCL INFO NCCL git version unknown unknown
49651bd10af9:650:650 [1] NCCL INFO NET/Plugin: Could not find: libnccl-net.so
49651bd10af9:650:650 [1] NCCL INFO NET/IB : No device found.
49651bd10af9:650:650 [1] NCCL INFO NET/IB : Using [RO]; OOB eth0:172.19.2.2<0>
49651bd10af9:650:650 [1] NCCL INFO Failed to initialize NET plugin IB
49651bd10af9:650:650 [1] NCCL INFO NET/Socket : Using [0]eth0:172.19.2.2<0>
49651bd10af9:650:650 [1] NCCL INFO Initialized NET plugin Socket
49651bd10af9:650:650 [1] NCCL INFO Assigned NET plugin Socket to comm
49651bd10af9:650:650 [1] NCCL INFO RMA/Plugin: Could not find: libnccl-rma.so
49651bd10af9:650:650 [1] NCCL INFO RMA/Plugin: Failed to initialize any plugin
49651bd10af9:650:650 [1] NCCL INFO GIN/Plugin: Could not find: libnccl-gin.

In [16]:
D0, D1 = 'cuda:0', 'cuda:1'

class ViTFront(nn.Module):
    def __init__(s, v):
        super().__init__()
        s.conv_proj, s.class_token = v.conv_proj, v.class_token
        s.pos, s.drop = v.encoder.pos_embedding, v.encoder.dropout
        s.blocks = nn.Sequential(*list(v.encoder.layers)[:6])
        s.h = v.hidden_dim
    def forward(s, x):
        n = x.shape[0]
        x = s.conv_proj(x).reshape(n, s.h, -1).permute(0, 2, 1)
        x = torch.cat([s.class_token.expand(n, -1, -1), x], dim=1) + s.pos
        return s.blocks(s.drop(x))

class ViTBack(nn.Module):
    def __init__(s, v):
        super().__init__()
        s.blocks = nn.Sequential(*list(v.encoder.layers)[6:])
        s.ln, s.heads = v.encoder.ln, v.heads
    def forward(s, x):
        return s.heads(s.ln(s.blocks(x))[:, 0])

def split(m, name):
    """Returns (part1, part2), sharing the original model's layers."""
    if name.startswith('resnet'):
        p1 = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2)
        p2 = nn.Sequential(m.layer3, m.layer4, m.avgpool, nn.Flatten(1), m.fc)
    else:
        p1, p2 = ViTFront(m), ViTBack(m)
    return p1, p2

# verify: the split model must give the same output and contain every parameter
for name in MODELS:
    m = build_model(name).to(D0).eval()
    p1, p2 = split(m, name)
    x, _ = make_batch(4, D0)
    with torch.no_grad():
        same = torch.allclose(m(x), p2(p1(x)), atol=1e-4)
    n_full = sum(p.numel() for p in m.parameters())
    n_split = sum(p.numel() for p in p1.parameters()) + sum(p.numel() for p in p2.parameters())
    print(f"{name:9s} same output: {same} | all params: {n_full == n_split}")
    del m, p1, p2
torch.cuda.empty_cache()

resnet18  same output: True | all params: True
resnet34  same output: True | all params: True
resnet50  same output: True | all params: True
vittiny   same output: True | all params: True
vitbase   same output: True | all params: True


In [17]:
def sync_both():
    torch.cuda.synchronize(D0); torch.cuda.synchronize(D1)

def bench_mp(model_name, bs):
    path = run_path(model_name, 'mp', bs)
    if already_done(path): return
    torch.cuda.empty_cache()

    p1, p2 = split(build_model(model_name), model_name)
    p1, p2 = p1.to(D0).train(), p2.to(D1).train()
    opt = torch.optim.SGD(list(p1.parameters()) + list(p2.parameters()), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, D0); y = y.to(D1)

    def natural():                       # a normal MP training step
        opt.zero_grad(set_to_none=True)
        crit(p2(p1(x).to(D1)), y).backward()
        opt.step()

    def natural_timed():
        sync_both(); t0 = time.perf_counter()
        natural()
        torch.cuda.synchronize(D0); ta = time.perf_counter()   # old pitfall: GPU 0 only
        torch.cuda.synchronize(D1); tb = time.perf_counter()   # correct: both GPUs
        return (tb - t0) * 1e3, (ta - t0) * 1e3

    def staged():                        # same step, split into 7 timed phases
        opt.zero_grad(set_to_none=True)
        t = []
        def mark(): sync_both(); t.append(time.perf_counter())
        mark()
        a = p1(x);                                mark()   # forward, GPU 0
        b = a.detach().to(D1).requires_grad_();   mark()   # forward transfer
        loss = crit(p2(b), y);                    mark()   # forward, GPU 1
        loss.backward();                          mark()   # backward, GPU 1
        g = b.grad.to(D0);                        mark()   # backward transfer
        a.backward(g);                            mark()   # backward, GPU 0
        opt.step();                               mark()   # optimizer
        return [(t[i + 1] - t[i]) * 1e3 for i in range(7)], a.numel() * 4 / 2**20

    t_w, n = time.perf_counter(), 0
    while n < WARMUP or time.perf_counter() - t_w < WARM_SEC:
        natural(); n += 1
    sync_both()
    for d in (D0, D1): torch.cuda.reset_peak_memory_stats(d)

    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows = []
    for i in range(MEASURED):
        if i % 2 == 0:
            nat, nat0 = natural_timed(); ph, act_mb = staged()
        else:
            ph, act_mb = staged(); nat, nat0 = natural_timed()
        rows.append([model_name, 'mp', bs, RESOLUTION, REPEAT, i, i % 2]
                    + [round(v, 3) for v in ph] + [round(sum(ph), 3), round(nat, 3), round(nat0, 3)])
    stop_clock_log(p, f)
    pk0, pk1 = (torch.cuda.max_memory_allocated(d) / 2**20 for d in (D0, D1))

    tmp = path + '.tmp'
    with open(tmp, 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['model', 'config', 'batch', 'res', 'repeat', 'step', 'order',
                    'fwd0_ms', 'xfer_fwd_ms', 'fwd1_ms', 'bwd1_ms', 'xfer_bwd_ms', 'bwd0_ms',
                    'opt_ms', 'staged_total_ms', 'step_ms', 'step_sync0_ms',
                    'act_mb', 'peak0_mb', 'peak1_mb'])
        for r in rows: w.writerow(r + [round(act_mb, 2), round(pk0, 1), round(pk1, 1)])
    os.replace(tmp, path)

    df = pd.DataFrame(rows).iloc[:, 7:].median()
    print(f"{tag}: step {df.iloc[8]:.1f} ms | GPU0 {df.iloc[0] + df.iloc[5]:.1f} | "
          f"GPU1 {df.iloc[2] + df.iloc[3]:.1f} | xfer {df.iloc[1] + df.iloc[4]:.2f} ms | "
          f"sync0-only error {100 * (1 - df.iloc[9] / df.iloc[8]):.1f}% | warm-up {n} steps")
    del p1, p2, opt, x, y
    torch.cuda.empty_cache()

In [18]:
for m in MODELS:
    for bs in BATCH_SIZES:
        bench_mp(m, bs)

resnet18-mp-bs16-res224-r1: step 43.9 ms | GPU0 28.0 | GPU1 15.1 | xfer 1.45 ms | sync0-only error 0.1% | warm-up 1364 steps
resnet18-mp-bs32-res224-r1: step 86.3 ms | GPU0 56.5 | GPU1 27.6 | xfer 2.80 ms | sync0-only error 0.1% | warm-up 710 steps
resnet18-mp-bs64-res224-r1: step 174.3 ms | GPU0 115.4 | GPU1 53.3 | xfer 5.43 ms | sync0-only error 0.0% | warm-up 353 steps
resnet34-mp-bs16-res224-r1: step 71.8 ms | GPU0 41.2 | GPU1 29.6 | xfer 1.47 ms | sync0-only error 0.0% | warm-up 841 steps
resnet34-mp-bs32-res224-r1: step 139.2 ms | GPU0 83.9 | GPU1 52.5 | xfer 2.77 ms | sync0-only error 0.0% | warm-up 433 steps
resnet34-mp-bs64-res224-r1: step 290.9 ms | GPU0 173.8 | GPU1 110.0 | xfer 5.39 ms | sync0-only error 0.0% | warm-up 209 steps
resnet50-mp-bs16-res224-r1: step 144.1 ms | GPU0 76.9 | GPU1 60.6 | xfer 5.37 ms | sync0-only error 0.0% | warm-up 420 steps
resnet50-mp-bs32-res224-r1: step 308.1 ms | GPU0 164.9 | GPU1 132.2 | xfer 10.57 ms | sync0-only error 0.0% | warm-up 197 st